# PS06 Visual Robustness Notebook
© 2026 ETH Zurich, Niclas Scheuer, Dejan Milojevic; Institute for Dynamic Systems and Control; Prof. Emilio Frazzoli

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

## Example 1: Linked Ellipse and 2x2 Step Response

Interpretation prompt:
- Select which input channel gets a step.
- On the left: the selected input direction (unit vector) maps through the same matrix $A$ to a steady output vector on the ellipse.
- On the right: $y_1(t), y_2(t)$ rise toward that mapped steady output.

This makes the geometric picture operational: the ellipse is the set of steady-state outputs for all unit input directions.

In [ ]:
def linked_2x2_step_ellipse_demo(
    theta_deg=30.0,
    sigma_max=2.5,
    sigma_min=0.7,
    input_case="u1 step",
    tau=1.0,
    t_snap=2.5,
):
    theta = np.deg2rad(theta_deg)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    S = np.diag([sigma_max, sigma_min])
    A = R @ S @ R.T

    # Input direction (unit vectors for channel-wise steps plus one combined direction).
    if input_case == "u1 step":
        u_dir = np.array([1.0, 0.0])
    elif input_case == "u2 step":
        u_dir = np.array([0.0, 1.0])
    else:
        u_dir = np.array([1.0, 1.0]) / np.sqrt(2.0)

    y_ss = A @ u_dir

    # First-order rise to the same steady-state map y_ss = A u_dir.
    t = np.linspace(0.0, 8.0, 500)
    alpha = 1.0 - np.exp(-t / max(tau, 1e-6))
    y = np.outer(alpha, y_ss)

    alpha_snap = 1.0 - np.exp(-t_snap / max(tau, 1e-6))
    y_snap = alpha_snap * y_ss

    # Ellipse from all unit input directions under y = A u.
    ang = np.linspace(0, 2 * np.pi, 400)
    circle = np.vstack((np.cos(ang), np.sin(ang)))
    ellipse = A @ circle

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    # Left: geometric mapping view.
    ax = axes[0]
    ax.plot(circle[0], circle[1], "k--", label="Unit input circle")
    ax.plot(ellipse[0], ellipse[1], color="tab:blue", linewidth=2, label="Mapped outputs (ellipse)")

    ax.arrow(0, 0, u_dir[0], u_dir[1], color="tab:green", head_width=0.06, length_includes_head=True)
    ax.arrow(0, 0, y_ss[0], y_ss[1], color="tab:red", head_width=0.08, length_includes_head=True)

    ax.plot(y[:, 0], y[:, 1], color="tab:orange", linewidth=2, label="Output vector trajectory")
    ax.scatter([y_snap[0]], [y_snap[1]], color="tab:orange", s=45, zorder=3)

    lim = max(3.0, sigma_max + 0.8)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title("Geometric map: input direction -> output vector")
    ax.legend(loc="upper right")

    # Right: output channels over time.
    ax2 = axes[1]
    ax2.plot(t, y[:, 0], label="y1(t)", linewidth=2)
    ax2.plot(t, y[:, 1], label="y2(t)", linewidth=2)
    ax2.axhline(y_ss[0], linestyle="--", alpha=0.6, color="tab:blue", label="y1 steady")
    ax2.axhline(y_ss[1], linestyle="--", alpha=0.6, color="tab:orange", label="y2 steady")
    ax2.axvline(t_snap, linestyle=":", color="gray", label="snapshot time")
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Output")
    ax2.set_title("2x2 system response to selected step input")
    ax2.legend(loc="best")

    fig.suptitle("Same matrix A in both views: ellipse is steady-state map, step plot is time evolution", y=1.02)
    plt.tight_layout()
    plt.show()


interact(
    linked_2x2_step_ellipse_demo,
    theta_deg=FloatSlider(value=30, min=0, max=180, step=1, description="angle (deg)"),
    sigma_max=FloatSlider(value=2.5, min=1.0, max=4.0, step=0.1, description="max gain"),
    sigma_min=FloatSlider(value=0.7, min=0.2, max=1.0, step=0.05, description="min gain"),
    input_case=Dropdown(options=["u1 step", "u2 step", "45deg step"], value="u1 step", description="input"),
    tau=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1, description="time const"),
    t_snap=FloatSlider(value=2.5, min=0.0, max=8.0, step=0.1, description="snapshot t"),
);

## Example 2: Small-Gain Intuition with Stability Boundary

Interpretation prompt:
- Increase/decrease block gains $k_1, k_2$.
- Observe what happens when effective loop gain crosses the stability threshold.

In [ ]:
def small_gain_demo(k1=1.2, k2=1.0, t_end=8.0):
    t = np.linspace(0, t_end, 500)
    k = k1 * k2
    a = k - 1.0

    if abs(a) < 1e-6:
        y = k * t
    elif a > 0:
        y = (k / a) * (1 - np.exp(-a * t))
    else:
        y = (k / a) * (1 - np.exp(-a * t))

    stable = a > 0

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(t, y, linewidth=2, label="Closed-loop step response")
    ax.axhline(1.0, linestyle="--", color="gray", label="Reference")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Output")
    ax.set_title("Small-gain intuition: stability changes with effective loop gain")

    status = "STABLE" if stable else ("MARGINAL" if abs(a) < 1e-6 else "UNSTABLE")
    color = "#d7f5d7" if stable else ("#fff3cd" if abs(a) < 1e-6 else "#f8d7da")
    ax.text(
        0.02,
        0.95,
        f"k1*k2 = {k:.2f} | threshold > 1.00 | {status}",
        transform=ax.transAxes,
        va="top",
        bbox=dict(facecolor=color, edgecolor="none"),
    )
    ax.legend(loc="best")
    plt.show()


interact(
    small_gain_demo,
    k1=FloatSlider(value=1.2, min=0.1, max=2.5, step=0.05, description="k1"),
    k2=FloatSlider(value=1.0, min=0.1, max=2.5, step=0.05, description="k2"),
    t_end=FloatSlider(value=8.0, min=3.0, max=15.0, step=0.5, description="horizon"),
);

## Example 3: Disturbance Rejection Tradeoff (PI Controller)

Interpretation prompt:
- Increase $K_p, K_i$ and watch disturbance rejection improve.
- Then observe the cost in control effort and sensitivity to measurement noise.

In [ ]:
def disturbance_tradeoff_demo(Kp=1.8, Ki=0.8, noise_amp=0.05):
    dt = 0.02
    T = 30.0
    t = np.arange(0.0, T, dt)

    x = 0.0
    integral_e = 0.0

    y_arr = np.zeros_like(t)
    u_arr = np.zeros_like(t)
    d_arr = np.zeros_like(t)

    for i, ti in enumerate(t):
        d = 1.0 if ti >= 8.0 else 0.0
        noise = noise_amp * np.sin(30.0 * ti)

        y_meas = x + noise
        e = 0.0 - y_meas
        integral_e += e * dt
        u = Kp * e + Ki * integral_e

        xdot = -x + u + d
        x += dt * xdot

        y_arr[i] = x
        u_arr[i] = u
        d_arr[i] = d

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    axes[0].plot(t, y_arr, label="Output y(t)", linewidth=2)
    axes[0].plot(t, d_arr, "--", label="Disturbance d(t)", alpha=0.8)
    axes[0].set_ylabel("Output / Disturbance")
    axes[0].set_title("Disturbance rejection tradeoff")
    axes[0].legend(loc="best")

    axes[1].plot(t, u_arr, color="tab:red", label="Control effort u(t)", linewidth=2)
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Control effort")
    axes[1].legend(loc="best")

    effort_rms = np.sqrt(np.mean(u_arr**2))
    axes[1].text(
        0.02,
        0.92,
        f"RMS control effort: {effort_rms:.3f}",
        transform=axes[1].transAxes,
        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
    )

    plt.tight_layout()
    plt.show()


interact(
    disturbance_tradeoff_demo,
    Kp=FloatSlider(value=1.8, min=0.2, max=4.0, step=0.1, description="Kp"),
    Ki=FloatSlider(value=0.8, min=0.0, max=2.5, step=0.05, description="Ki"),
    noise_amp=FloatSlider(value=0.05, min=0.0, max=0.2, step=0.01, description="noise"),
);

## Example 4: Frequency-Weight Shaping Intuition

Interpretation prompt:
- Tune low-frequency and high-frequency weighting priorities.
- Observe how the desired loop shape changes (tracking/disturbance rejection vs noise attenuation).

Short guide:
- Low frequencies (left side of the plot): mainly about tracking and disturbance rejection.
- High frequencies (right side of the plot): mainly about noise attenuation and avoiding aggressive control action.

In [ ]:
def weight_shape_demo(K=6.0, wb_low=0.5, wb_high=5.0, A_low=15.0, A_high=20.0):
    w = np.logspace(-2, 2, 400)

    # Conceptual low-frequency priority curve: high at low w, relaxes toward 1.
    W_low = 1.0 + A_low / (1.0 + (w / wb_low) ** 2)

    # Conceptual high-frequency penalty: near 1 at low w, rises at high w.
    W_high = 1.0 + A_high * ((w / wb_high) ** 2 / (1.0 + (w / wb_high) ** 2))

    # Simple open-loop template magnitude (for visual comparison only).
    p1, p2 = 0.8, 7.0
    L_mag = K / np.sqrt((1.0 + (w / p1) ** 2) * (1.0 + (w / p2) ** 2))

    lower_target = W_low
    upper_target = 1.0 / W_high

    # Heuristic checks for qualitative feedback.
    low_band = w <= max(1.0, wb_low)
    high_band = w >= max(1.0, wb_high)

    low_margin_db = np.min(20 * np.log10(L_mag[low_band] / lower_target[low_band]))
    high_margin_db = np.min(20 * np.log10(upper_target[high_band] / L_mag[high_band]))

    dr_good = low_margin_db >= 0.0
    noise_good = high_margin_db >= 0.0

    if dr_good and noise_good:
        verdict = "Good balance: disturbance rejection and noise attenuation are both satisfactory."
    elif dr_good and not noise_good:
        verdict = "Disturbance rejection is good, but high-frequency noise attenuation is weak."
    elif not dr_good and noise_good:
        verdict = "Noise attenuation is good, but low-frequency disturbance rejection is weak."
    else:
        verdict = "Both are weak: increase low-frequency gain and/or reduce high-frequency loop gain."

    fig, ax = plt.subplots(figsize=(10, 5.2))
    ax.semilogx(w, 20 * np.log10(L_mag), label="|L(jw)| template", linewidth=2)
    ax.semilogx(w, 20 * np.log10(lower_target), "--", label="Low-freq priority (conceptual lower target)")
    ax.semilogx(w, 20 * np.log10(upper_target), "--", label="High-freq attenuation target")

    ax.set_title("Frequency-shaping intuition: balancing low- and high-frequency priorities")
    ax.set_xlabel("Frequency (rad/s)")
    ax.set_ylabel("Magnitude (dB)")
    ax.legend(loc="best")

    plt.tight_layout()
    plt.show()

    print(f"Assessment: {verdict}")
    print(
        f"Low-frequency margin: {low_margin_db:+.1f} dB | "
        f"High-frequency margin: {high_margin_db:+.1f} dB"
    )


interact(
    weight_shape_demo,
    K=FloatSlider(value=6.0, min=1.0, max=20.0, step=0.5, description="loop gain K"),
    wb_low=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description="wb low"),
    wb_high=FloatSlider(value=5.0, min=1.0, max=30.0, step=0.5, description="wb high"),
    A_low=FloatSlider(value=15.0, min=0.0, max=40.0, step=1.0, description="low weight"),
    A_high=FloatSlider(value=20.0, min=0.0, max=40.0, step=1.0, description="high penalty"),
);